In [16]:
# -*- coding: utf-8 -*-
import os
import re
import time
import requests
from typing import List, Dict

In [28]:
# =============== 配置 ===============
ANKI_URL = "http://127.0.0.1:8765"
DEEPSEEK_API_URL = "https://api.deepseek.com/chat/completions"
DEEPSEEK_API_KEY = os.getenv("DEEPSEEK_API_KEY") 

DECK_NAME = "Everyday English"
MODEL_NAME = "newFastWQ"
DEFAULT_TAGS = ["Everyday English", "newFastWQ", "flomo", "deepseek"]

In [4]:
# =============== Anki-Connect 通用封装 ===============
def _ac_request(action, **params):
    return {"action": action, "params": params, "version": 6}

def ac_invoke(action, **params):
    payload = _ac_request(action, **params)
    try:
        resp = requests.post(ANKI_URL, json=payload, timeout=10)
    except requests.exceptions.RequestException as e:
        raise SystemExit(f"❌ 无法连接 Anki-Connect：{e}")
    if resp.status_code != 200:
        raise RuntimeError(f"HTTP {resp.status_code}: {resp.text}")
    data = resp.json()
    if not isinstance(data, dict) or "error" not in data or "result" not in data:
        raise RuntimeError(f"响应格式异常：{data}")
    if data["error"] is not None:
        raise RuntimeError(f"Anki-Connect 错误：{data['error']}")
    return data["result"]

def ensure_deck(deck_name: str):
    decks = set(ac_invoke("deckNames"))
    if deck_name not in decks:
        ac_invoke("createDeck", deck=deck_name)

def get_model_fields(model_name: str) -> List[str]:
    return ac_invoke("modelFieldNames", modelName=model_name)

In [22]:
# ====================== 字段构建 ======================
def build_fields(model_fields: List[str], provided: Dict) -> Dict:
    """
    支持两种键：
    1) 精确字段名（若知道 model 的字段，就直接用同名键）
    2) 通用键：term / phonetic / gloss / example / collocation
    """
    fields = {f: "" for f in model_fields}

    # 显式同名字段优先
    for k, v in provided.items():
        if k in fields:
            fields[k] = v

    # 通用键 → 常见字段名的映射（仅填空位）
    def _fill_generic(key, candidates):
        if key in provided:
            for cand in candidates:
                if cand in fields and not fields[cand]:
                    fields[cand] = provided[key]
                    break

    _fill_generic("term",       ["Word", "Expression", "Term", "Headword", "Front", "单词", "词条"])
    _fill_generic("phonetic",   ["Phonetic", "IPA", "音标"])
    _fill_generic("gloss",      ["Definition", "Meaning", "Gloss", "释义", "中文释义", "词典释义", "Back"])
    _fill_generic("example",    ["Sentence", "Example", "Examples", "例句"])
    _fill_generic("collocation",["Collocation", "Phrases", "搭配"])

    return fields

def add_note(
    deck_name: str,
    model_name: str,
    provided_fields: dict | None = None,
    tags: list[str] | None = None,
    audio: dict | None = None,
    picture: dict | None = None,
) -> int:
    """
    :param provided_fields: 既可用“模型同名字段”，也可用通用键：
        term / gloss / example / phonetic / collocation
        （例如 provided_fields = {"term":"aurora","gloss":"极光","example":"We saw ..."}）
    :param audio: 可选，如 {"url": "...mp3", "filename": "aurora.mp3", "fields": ["Sound"]}
    :param picture: 可选，如 {"url": "...jpg", "filename": "aurora.jpg", "fields": ["Image"]}
    :return: 新增笔记的 noteId
    """
    provided_fields = provided_fields or {}
    tags = tags

    # 1) 确保牌组存在
    ensure_deck(deck_name)

    # 2) 获取模型字段并构造字段映射
    model_fields = get_model_fields(model_name)  # 若模型不存在，这里会抛出异常
    fields = build_fields(model_fields, provided_fields)

    # 3) 组装 note
    note = {
        "deckName": deck_name,
        "modelName": model_name,
        "fields": fields,
        "tags": tags,
        "options": {
            "allowDuplicate": False,
            "duplicateScope": "deck",  # 在本牌组内查重
        },
    }

    # 附件（可选）
    if audio:
        note["audio"] = [{
            "url": audio["url"],
            "filename": audio["filename"],
            "fields": audio.get("fields", []),
        }]
    if picture:
        note["picture"] = [{
            "url": picture["url"],
            "filename": picture["filename"],
            "fields": picture.get("fields", []),
        }]

    # 4) 调用 addNote
    note_id = ac_invoke("addNote", note=note)
    return note_id

In [25]:
# =============== 词串拆分 ===============
def split_words_cn_comma(s: str) -> List[str]:
    """
    把以中文逗号（，）为主、兼容英文逗号/顿号/空白的词串拆分为词列表。
    例： "aurora，photosynthesis，metabolism" → ["aurora","photosynthesis","metabolism"]
    """
    if not s:
        return []
    parts = re.split(r"，+", s.strip())
    return [p for p in (x.strip() for x in parts) if p]

In [7]:
# =============== DeepSeek：生成英文例句 ===============
def deepseek_example_for(word: str) -> str:
    """
    用 DeepSeek 生成一个地道、简洁的英文例句（<=16词），包含该单词。
    返回纯句子（不带引号/标号）。
    """
    if not DEEPSEEK_API_KEY:
        raise RuntimeError("缺少 DEEPSEEK_API_KEY 环境变量。")

    prompt = (
        "You are an English writing tutor. "
        f"Write ONE natural, common English sentence (≤16 words) that uses '{word}' in a typical context. "
        "Return ONLY the sentence. No quotes, labels, or extra text."
    )
    headers = {
        "Authorization": f"Bearer {DEEPSEEK_API_KEY}",
        "Content-Type": "application/json",
    }
    body = {
        "model": "deepseek-chat",
        "messages": [
            {"role": "system", "content": "You produce concise, natural English sentences."},
            {"role": "user", "content": prompt}
        ],
        "temperature": 0.7,
        "max_tokens": 64,
        "top_p": 0.9,
    }
    try:
        resp = requests.post(DEEPSEEK_API_URL, json=body, headers=headers, timeout=20)
        resp.raise_for_status()
        data = resp.json()
        text = data["choices"][0]["message"]["content"].strip()
        # 清理潜在引号/多余标点/换行
        text = text.strip().strip('"').strip("'").replace("\n", " ").strip()
        # 确保包含目标词（如不含就轻度兜底补一个简单句）
        if word.lower() not in text.lower():
            text = f"I often use the word '{word}' in daily conversations."
        return text
    except requests.RequestException as e:
        raise RuntimeError(f"DeepSeek 请求失败：{e}")
    except Exception as e:
        raise RuntimeError(f"DeepSeek 解析失败：{e}")

In [30]:
# =============== 批量处理主流程 ===============
def add_words_with_examples(
    words_cn_comma: str,
    deck_name: str = DECK_NAME,
    model_name: str = MODEL_NAME,
    sleep_between: float = 0.6,
):
    """
    1) 把中文逗号分隔的词串拆分成列表
    2) 对每个词调用 DeepSeek 生成英文例句
    3) 添加到 Anki，字段采用通用键映射：term + example
    """
    words = split_words_cn_comma(words_cn_comma)
    if not words:
        print("⚠️ 输入为空，未处理。")
        return

    print(f"📝 待添加词数：{len(words)} -> {words}")
    ok, fail = 0, 0
    for w in words:
        try:
            ex = deepseek_example_for(w)
            note_id = add_note(
                deck_name=deck_name,
                model_name=model_name,
                provided_fields={
                    "term": w,
                    "example": ex,
                },
                tags=DEFAULT_TAGS,
            )
            ok += 1
            print(f"✅ {w} -> noteId={note_id} | example: {ex}")
        except Exception as e:
            fail += 1
            print(f"❌ {w} 失败：{e}")
        # 简单限速，避免触发速率限制
        time.sleep(sleep_between)

    print(f"\n完成：成功 {ok}，失败 {fail}，总计 {len(words)}")

In [32]:
input_words = "campfire，memorable，cardboard，gala，inspired，parade，eccentric，nachos，legacy"
add_words_with_examples(input_words)

📝 待添加词数：9 -> ['campfire', 'memorable', 'cardboard', 'gala', 'inspired', 'parade', 'eccentric', 'nachos', 'legacy']
✅ campfire -> noteId=1759798718503 | example: We gathered around the campfire to tell stories.
✅ memorable -> noteId=1759798721123 | example: That vacation was truly memorable for our whole family.
✅ cardboard -> noteId=1759798723894 | example: We need more cardboard boxes for the move next week.
✅ gala -> noteId=1759798726769 | example: We're going to the annual charity gala this weekend.
✅ inspired -> noteId=1759798729231 | example: Her speech inspired me to pursue my dreams.
✅ parade -> noteId=1759798731686 | example: The parade will start at noon on Main Street.
✅ eccentric -> noteId=1759798734360 | example: My neighbor is known for his eccentric collection of garden gnomes.
✅ nachos -> noteId=1759798736935 | example: I ordered a large plate of nachos to share with my friends.
✅ legacy -> noteId=1759798739790 | example: His legacy will be remembered for generations to 